# Podstawy: pandas ↔ polars ↔ DuckDB

**Problem:** te trzy narzędzia rozwiązują ten sam zestaw podstawowych zadań (wczytaj dane, wybierz kolumny, odfiltruj wiersze, policz agregat) trzema różnymi stylami składni — obiektowym (pandas), wyrażeniowym (polars) i SQL-owym (DuckDB). Ta notatka to "kamień z Rosetty" — każda sekcja pokazuje to samo zadanie we wszystkich trzech.

**Porównanie:**
- **pandas** — najbardziej rozpowszechniony, ogromny ekosystem, ale wolniejszy na dużych danych i z historycznymi pułapkami (kopie vs widoki).
- **polars** — API wyrażeniowe (`pl.col(...)`), szybszy silnik (Rust), wbudowany tryb `lazy` z optymalizacją zapytań.
- **DuckDB** — silnik SQL działający bezpośrednio na plikach (CSV/Parquet) lub na istniejących DataFrame'ach pandas/polars **bez ich importowania** — dobry pierwszy krok w stronę BigQuery, bo składnia SQL jest niemal identyczna.

**Kiedy stosować:** pandas jako domyślny wybór przy dobrze znanym API i mniejszych danych; polars przy większych danych lub gdy liczy się wydajność; DuckDB, gdy naturalniej myśli się w SQL albo trzeba odpytać plik bez wczytywania go w całości do pamięci.

**Zakres:** wczytywanie danych, inspekcja, select, filtrowanie, kolumny, sortowanie, prosta agregacja, zmiana nazw, duplikaty, braki danych, zapis. Grupowanie, pivotowanie, okna czasowe i złączenia mają osobne, głębsze notatki w tym repo — tu tylko podstawy.

## Setup

In [ ]:
import pandas as pd
import polars as pl
import duckdb
import numpy as np

rng = np.random.default_rng(3)
n = 12
data = {
    "region": ["North", "North", "South", "South", "East", "East",
               "West", "West", "North", "South", "East", "West"],
    "category": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "A", "B", "B"],
    "sales": rng.integers(500, 2000, n),
    "units": rng.integers(5, 50, n),
}
pd.DataFrame(data).to_csv("sales.csv", index=False)
print("Zapisano sales.csv")

## Sekcja 1 — Wczytywanie danych

**Trik warty zapamiętania:** DuckDB potrafi odpytać SQL-em istniejący obiekt pandas/polars DataFrame **bezpośrednio po nazwie zmiennej Pythona** — bez żadnego importu czy rejestracji.

In [ ]:
# pandas
df = pd.read_csv("sales.csv")

# polars
df_pl = pl.read_csv("sales.csv")

# DuckDB - wczytanie z pliku
duckdb.sql("SELECT * FROM read_csv('sales.csv') LIMIT 3")

In [ ]:
# DuckDB - trik: SQL bezpośrednio na już istniejącym df / df_pl, po nazwie zmiennej
duckdb.sql("SELECT * FROM df LIMIT 3")

## Sekcja 2 — Podstawowa inspekcja

In [ ]:
print(df.shape, "|", dict(df.dtypes))
print(df_pl.shape, "|", df_pl.schema)
duckdb.sql("DESCRIBE SELECT * FROM df")

## Sekcja 3 — Wybór kolumn

In [ ]:
df[["region", "sales"]].head(2)

In [ ]:
df_pl.select("region", "sales").head(2)

In [ ]:
duckdb.sql("SELECT region, sales FROM df LIMIT 2")

## Sekcja 4 — Filtrowanie wierszy

In [ ]:
# pandas - maska boolowska
df[(df["sales"] > 1000) & (df["region"] == "North")]

In [ ]:
# polars - .filter() z wyrażeniami
df_pl.filter((pl.col("sales") > 1000) & (pl.col("region") == "North"))

In [ ]:
# DuckDB - WHERE, dokładnie jak w zwykłym SQL
duckdb.sql("SELECT * FROM df WHERE sales > 1000 AND region = 'North'")

## Sekcja 5 — Dodawanie / modyfikacja kolumn

In [ ]:
df.assign(sales_per_unit=df["sales"] / df["units"]).head(2)

In [ ]:
df_pl.with_columns((pl.col("sales") / pl.col("units")).alias("sales_per_unit")).head(2)

In [ ]:
duckdb.sql("SELECT *, sales / units AS sales_per_unit FROM df LIMIT 2")

## Sekcja 6 — Sortowanie

In [ ]:
df.sort_values("sales", ascending=False).head(2)

In [ ]:
df_pl.sort("sales", descending=True).head(2)

In [ ]:
duckdb.sql("SELECT * FROM df ORDER BY sales DESC LIMIT 2")

## Sekcja 7 — Prosta agregacja (bez `groupby`)

Agregacja całej kolumny do pojedynczej liczby. Grupowanie po kategoriach (`groupby`/`GROUP BY`) ma osobną, dużo głębszą notatkę w tym repo — tu tylko najprostszy przypadek.

In [ ]:
print(df["sales"].sum(), df["sales"].mean())
print(df_pl["sales"].sum(), df_pl["sales"].mean())
duckdb.sql("SELECT SUM(sales), AVG(sales) FROM df")

## Sekcja 8 — Zmiana nazw kolumn

In [ ]:
print(df.rename(columns={"sales": "revenue"}).columns.tolist())
print(df_pl.rename({"sales": "revenue"}).columns)
duckdb.sql("SELECT sales AS revenue FROM df LIMIT 1")

## Sekcja 9 — Duplikaty i braki danych

In [ ]:
df_dup = pd.concat([df, df.iloc[[0]]], ignore_index=True)  # sztucznie dodany duplikat
print(f"pandas - zduplikowane wiersze: {df_dup.duplicated().sum()}, po drop_duplicates: {df_dup.drop_duplicates().shape}")

df_pl_dup = pl.concat([df_pl, df_pl[[0]]])
print(f"polars - zduplikowane wiersze: {df_pl_dup.is_duplicated().sum()}, po unique: {df_pl_dup.unique().shape}")

In [ ]:
df_na = df.copy()
df_na.loc[0, "sales"] = np.nan

print(df_na.isna().sum())
print(f"\nPo dropna: {df_na.dropna().shape}")
print(f"Po fillna(0), pierwszy wiersz sales: {df_na.fillna(0).loc[0, 'sales']}")

## Sekcja 10 — `value_counts` / zliczanie unikalnych wartości

In [ ]:
print(df["region"].value_counts())
print()
print(df_pl["region"].value_counts())
print()
duckdb.sql("SELECT region, COUNT(*) AS n FROM df GROUP BY region ORDER BY n DESC")

## Sekcja 11 — Materializacja wyniku DuckDB do pandas/polars

`duckdb.sql(...)` domyślnie zwraca leniwą relację (`DuckDBPyRelation`) — dopiero `.df()` lub `.pl()` zamienia wynik na konkretny, materialny obiekt pandas/polars, żeby przekazać go dalej do reszty pipeline'u.

In [ ]:
result = duckdb.sql("SELECT region, SUM(sales) AS total FROM df GROUP BY region")
print(type(result))

as_pandas = result.df()
as_polars = result.pl()
print(type(as_pandas), type(as_polars))

## Sekcja 12 — Zapis danych

In [ ]:
df.to_csv("out_pandas.csv", index=False)
df_pl.write_csv("out_polars.csv")
duckdb.sql("COPY df TO 'out_duckdb.csv' (HEADER, DELIMITER ',')")
print("Zapisano trzy pliki wyjściowe")

## Sekcja 13 — Pułapki

### Pułapka 1 — pandas: pojedynczy `[]` zwraca `Series`, podwójny `[[]]` zwraca `DataFrame`

To dwa różne typy z różnym zestawem metod. `df['sales']` nie ma np. `.columns` — łatwo o to potknąć się przy pisaniu funkcji, która ma działać na obu.

In [ ]:
single = df["sales"]
double = df[["sales"]]
print(f"df['sales']   -> {type(single).__name__}")
print(f"df[['sales']] -> {type(double).__name__}")

try:
    single.columns
except AttributeError as e:
    print(f"single.columns -> błąd: {e}")

### Pułapka 2 — polars: `lazy()` buduje PLAN, nie wynik

Wersja `eager` (domyślna, `pl.DataFrame`) wykonuje operacje natychmiast. Wersja `lazy` (`.lazy()` albo `pl.scan_csv(...)` zamiast `pl.read_csv(...)`) buduje plan zapytania i optymalizuje go dopiero przy `.collect()` — do tego momentu `print()` pokazuje sam plan, nie dane. Częsty błąd początkujących: zapomnienie `.collect()` i zdziwienie, że "nic nie ma" albo że wynik to nieczytelny plan zamiast tabeli.

In [ ]:
lazy_query = df_pl.lazy().filter(pl.col("sales") > 1000).select("region", "sales")
print(f"Typ przed collect(): {type(lazy_query).__name__}")
print(lazy_query)  # plan zapytania, NIE dane

print("\nPo .collect():")
collected = lazy_query.collect()
print(f"Typ: {type(collected).__name__}")
collected

### Pułapka 3 — pandas: łańcuchowe przypisanie (`SettingWithCopyWarning`)

Klasyczna pułapka: przypisanie do wyniku filtrowania (`df[maska]["kolumna"] = wartość`) może modyfikować tymczasową kopię zamiast oryginału — historycznie z ostrzeżeniem `SettingWithCopyWarning`. Od pandas 3.0 domyślny mechanizm *Copy-on-Write* czyni to zachowanie bezpieczniejszym (mniej cichych pomyłek), ale we wcześniejszych wersjach (1.x/2.x, wciąż powszechnych w firmowych środowiskach) ten wzorzec bywał realnie zawodny. Bezpieczny nawyk niezależny od wersji: `.copy()` przy tworzeniu podzbioru, `.loc[]` przy przypisaniu.

In [ ]:
# Bezpieczny wzorzec, działający niezależnie od wersji pandas
subset = df[df["region"] == "North"].copy()
subset.loc[:, "sales"] = 0
subset

## Podsumowanie

| Zadanie | pandas | polars | DuckDB |
|---|---|---|---|
| Wczytanie CSV | `pd.read_csv("f.csv")` | `pl.read_csv("f.csv")` | `duckdb.sql("SELECT * FROM read_csv('f.csv')")` |
| Odpytanie już istniejącego df/df_pl | nie dotyczy | nie dotyczy | `duckdb.sql("SELECT * FROM df")` (po nazwie zmiennej!) |
| Kształt / typy kolumn | `.shape`, `.dtypes` | `.shape`, `.schema` | `DESCRIBE SELECT * FROM df` |
| Wybór kolumn | `df[["a","b"]]` | `df.select("a","b")` | `SELECT a, b FROM df` |
| Filtrowanie | `df[warunek]` | `df.filter(wyrażenie)` | `WHERE warunek` |
| Nowa/zmieniona kolumna | `df.assign(x=...)` | `df.with_columns(...)` | `SELECT *, wyrażenie AS x` |
| Sortowanie | `.sort_values("col")` | `.sort("col")` | `ORDER BY col` |
| Suma/średnia całej kolumny | `df["col"].sum()` | `df["col"].sum()` | `SELECT SUM(col) FROM df` |
| Zmiana nazwy kolumny | `.rename(columns={...})` | `.rename({...})` | `SELECT col AS nowa_nazwa` |
| Duplikaty | `.duplicated()` / `.drop_duplicates()` | `.is_duplicated()` / `.unique()` | `SELECT DISTINCT ...` |
| Braki danych | `.isna()`, `.dropna()`, `.fillna()` | `.null_count()`, `.drop_nulls()`, `.fill_null()` | `IS NULL`, `COALESCE(...)` |
| Liczba wystąpień wartości | `.value_counts()` | `.value_counts()` | `GROUP BY ... COUNT(*)` |
| Zapis do pliku | `.to_csv(...)` | `.write_csv(...)` | `COPY df TO 'f.csv'` |
| Wynik zapytania jako pandas/polars | nie dotyczy | nie dotyczy | `.df()` / `.pl()` na wyniku |

**Wniosek:** DuckDB jest szczególnie wygodnym punktem wejścia właśnie dlatego, że składnia `SELECT`/`WHERE`/`GROUP BY` jest niemal 1:1 z T-SQL — a możliwość odpytania istniejącego `df`/`df_pl` bez żadnej migracji danych czyni go naturalnym narzędziem do szybkiej weryfikacji, zanim coś trafi do właściwego pipeline'u BigQuery.